# Phase 35 v1.0f - Synthetic Interpretability Smoke
## Reviewer Notebook (read-only)

**Classification:** `READ_ONLY_PHASE35_V1_0F_SYNTHETIC_SMOKE_REVIEWER_NOTEBOOK`
**Revision:** `ra_medfilm_barr_phase35_interpretability_v1_0f`
**Smoke classification:** `PRE_SCIENTIFIC_SYNTHETIC_NUMERICAL_SAFETY_AND_INTERFACE_VALIDATION`
**Explicit:** `NOT_PHASE35_SCIENTIFIC_INTERPRETABILITY_RESULT`

This notebook reviews the ONE authorized v1.0f synthetic-only smoke execution. It is READ-ONLY over
the produced JSON artifacts. It does not construct models, load checkpoints, run forwards or
autograd, run Grad-CAM, run router attribution, read real medical data, or read TEST/HOLDOUT.

The smoke was invoked exactly ONCE (`--mode smoke`) with `PHASE35_SMOKE_EXECUTE=1` and
`PHASE35_EXECUTE` unset. No retry was performed.

## 0. Setup (read-only artifact loading)

In [1]:
import hashlib
import json
from pathlib import Path
ROOT = Path(r"D://Softfors//jup"); AUD = ROOT / "reports" / "audits"
def jload(p):
    return json.loads(Path(p).read_text(encoding="utf-8"))
def sha(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()
smoke = jload(AUD / "phase35_synthetic_interpretability_smoke_v1_0f.json")
prov  = jload(AUD / "phase35_synthetic_smoke_provenance_v1_0f.json")
fc    = jload(AUD / "phase35_synthetic_smoke_final_check_v1_0f.json")
term  = jload(AUD / "phase35_synthetic_smoke_terminal_v1_0f.json")
freeze = jload(AUD / "phase35_preexecution_source_freeze_v1_0f.json")
print("Phase35 v1.0f synthetic smoke review (READ-ONLY)")
print("This notebook READS the produced v1.0f smoke/provenance/final-check/terminal artifacts only.")
print("No model construction, no checkpoint deserialization, no forward, no autograd, no Grad-CAM,")
print("no router attribution, no images, no real p_global, no VAL808 cache, no HOLDOUT/TEST.")


Phase35 v1.0f synthetic smoke review (READ-ONLY)
This notebook READS the produced v1.0f smoke/provenance/final-check/terminal artifacts only.
No model construction, no checkpoint deserialization, no forward, no autograd, no Grad-CAM,
no router attribution, no images, no real p_global, no VAL808 cache, no HOLDOUT/TEST.


## 1. Result and blocking status

In [2]:
print("terminal:", term["terminal"])
print("smoke result:", smoke["result"])
print("harness all_pass:", smoke["harness_all_pass"])
print("counts_match_expected:", smoke["counts_match_expected"])
print("scientific_and_interface_checks_all_pass:", smoke["scientific_and_interface_checks_all_pass"])
print("invocation_count:", smoke["invocation_count"], "retries:", smoke["retries_performed"])
print("final check:", fc["n_checks"], "checks,", fc["n_failed"], "failed")
print("failed checks:", fc["failed"])

terminal: PHASE35_SYNTHETIC_SMOKE_V1_0F_BLOCKED__CANONICAL_MODEL_OR_NUMERICAL_INTERPRETABILITY_INTERFACE_CHECK_FAILED__ZERO_PROTECTED_DATA_OR_SCIENTIFIC_OUTPUT_ACCESS
smoke result: FAIL_CLOSED
harness all_pass: False
counts_match_expected: False
scientific_and_interface_checks_all_pass: True
invocation_count: 1 retries: 0
final check: 106 checks, 2 failed
failed checks: ['105_harness_all_pass_TRUE', '106_counts_match_expected_TRUE']


## 2. Blocking root cause

Every model, numerical, interface, and protected-data check passed. The single blocking condition is
an **internal execution-count integrity mismatch**: `counts_match_expected=False`, driven by the
counter `synthetic_smoke_target_masks` = **2** where the frozen v1.0c expected value is **1**.

Mechanism: the frozen v1.0c counter store is reused VERBATIM, but the v1.0f change introduced a
second increment. `V0D.smoke_synthetic_target_mask_rank2()` increments once for the primary
centered-square mask; `empty_mask_helper_check_insession_actual_logits()` then increments again for
the all-False empty variant derived from the SAME primary mask. `SMOKE_EXPECTED_COUNTERS` was not
updated alongside the repair, so the harness's internal consistency gate correctly fails closed.

In [3]:
m = smoke["counter_mismatches"]["synthetic_smoke_target_masks"]
for k in ("observed","expected","cause","semantic_effect","scientific_effect"):
    print(f"{k}: {m[k]}")
print()
print("blocking detail:", json.dumps(fc["blocking_detail"], indent=1))

observed: 2
expected: 1
cause: DOUBLE_INCREMENT: V0D.smoke_synthetic_target_mask_rank2() increments for the PRIMARY target mask, and the v1.0f in-scope empty-mask observability check (empty_mask_helper_check_insession_actual_logits, line 176 of the v1.0f source) increments AGAIN for the same primary mask's all-False empty variant.
semantic_effect: NONE: a secondary observability derivative of the SAME primary mask; the work order specifies primary_target_masks=1 (actual-logits empty-mask checks are a separate category), so the primary-target-mask execution count is 1 and remains compliant.
scientific_effect: NONE: no additional model construction, forward, autograd, CAM, router row, or protected-data access.

blocking detail: {
 "counter": "synthetic_smoke_target_masks",
 "observed": 2,
 "expected": 1,
 "cause": "DOUBLE_INCREMENT (V0D primary mask + v1.0f in-scope empty-mask observability)",
 "scientific_effect": "NONE",
 "protected_data_effect": "NONE"
}


## 3. Source / hash gate and environment

In [4]:
print("sources verified:", prov["source_hash_gate"]["n_sources_verified"],
      "mismatch:", prov["source_hash_gate"]["n_mismatch"],
      "before model construction:", prov["source_hash_gate"]["verified_before_model_construction"])
print("G checkpoint match:", prov["checkpoint_gate"]["G"]["match"], prov["checkpoint_gate"]["G"]["sha256"])
print("B_phi checkpoint match:", prov["checkpoint_gate"]["B_phi"]["match"], prov["checkpoint_gate"]["B_phi"]["sha256"])
env = smoke["environment"]
print("env:", env["python"], env["torch"], env["gpu"], "cc", env["cc"], "cuda", env["cuda_available"])

sources verified: 18 mismatch: 0 before model construction: True
G checkpoint match: True 286689fdd5c63ad75a383151d267af8b476e7a98327632de292ce6ed5b5ba1b7
B_phi checkpoint match: True 8a412b2fe82f225f844fda07379a6436c1d73d435f885b26a1d88d5b2f9fc43b
env: 3.11.9 2.6.0+cu124 NVIDIA GeForce RTX 3060 Ti cc 8.6 cuda True


## 4. Canonical model construction, strict load, and parameter counts

In [5]:
ms = smoke["model_scope"]
for k in ("G_constructions","B_phi_constructions","checkpoint_deserializations","strict_loads",
          "G_parameter_count","B_phi_parameter_count","G_missing_keys","G_unexpected_keys",
          "B_phi_missing_keys","B_phi_unexpected_keys","R_psi_construction_load_forward",
          "S_omega_construction_load_forward"):
    print(f"{k}: {ms[k]}")

G_constructions: 1
B_phi_constructions: 1
checkpoint_deserializations: 2
strict_loads: 2
G_parameter_count: 3523001
B_phi_parameter_count: 36801
G_missing_keys: []
G_unexpected_keys: []
B_phi_missing_keys: []
B_phi_unexpected_keys: []
R_psi_construction_load_forward: [0, 0, 0]
S_omega_construction_load_forward: [0, 0, 0]


## 5. FiLM sites and task embeddings

In [6]:
fs = smoke["film_smoke"]
print("film all_pass:", fs["all_pass"], "| scientific_film_tables_emitted:", fs["scientific_film_tables_emitted"])
sites = {}
for key, v in fs["per_task_site"].items():
    dom, site = key.split("|")
    sites.setdefault(site, set()).add((v["gamma_len"], v["expected_channels"], v["finite"]))
for site in ("bridge","decoder4","decoder3","decoder2","decoder1"):
    print(site, "gamma_len/expected/finite:", sites[site])
print()
es = smoke["embedding_smoke"]
print("G task embedding:", es["G_task_embedding_shape"], "finite:", es["G_finite"],
      "| B_phi task embedding:", es["B_phi_task_embedding_shape"], "finite:", es["B_phi_finite"])

film all_pass: True | scientific_film_tables_emitted: False
bridge gamma_len/expected/finite: {(256, 256, True)}
decoder4 gamma_len/expected/finite: {(192, 192, True)}
decoder3 gamma_len/expected/finite: {(96, 96, True)}
decoder2 gamma_len/expected/finite: {(48, 48, True)}
decoder1 gamma_len/expected/finite: {(24, 24, True)}

G task embedding: [4, 32] finite: True | B_phi task embedding: [4, 16] finite: True


## 6. Synthetic input and direct target-scalar diagnostics

In [7]:
si = smoke["synthetic_input"]
print("seed:", si["seed"], "image:", si["image_shape"], si["image_dtype"], si["image_device"],
      "| task_id:", si["task_id"], si["task_id_dtype"])
print("mask:", si["target_mask_name"], "rank", si["target_mask_rank"], si["target_mask_shape"],
      "true pixels:", si["target_mask_true_pixels"], "|", si["target_mask_centered_square"])
print()
ts = smoke["target_scalar_direct"]
print("impl:", ts["impl"])
print("value:", ts["value"], "finite:", ts["finite"], "selected_pixels:", ts["selected_pixels"])
print("surrogate_logits_used:", ts["surrogate_logits_used"])

seed: 42 image: [1, 3, 512, 512] float32 cuda | task_id: [0] int64
mask: SMOKE_SYNTHETIC_TARGET_MASK_RANK2 rank 2 [512, 512] true pixels: 16384 | rows 192:320, cols 192:320

impl: gradcam_target_scalar_rank_safe
value: -13.834142684936523 finite: True selected_pixels: 16384
surrogate_logits_used: False


## 7. Grad-CAM activations and gradients (bridge / decoder1)

In [8]:
gf = smoke["gradcam_forward"]
print("G forwards:", gf["G_forward_count"], "autograd calls:", gf["autograd_calls"])
print("activation shapes:", gf["activation_shapes"])
print("enable_grad:", gf["enable_grad"], "autocast_cuda_fp16:", gf["autocast_cuda_fp16"],
      "no_grad_used:", gf["no_grad_used"], "inference_mode_used:", gf["inference_mode_used"])
print()
for n in ("bridge","decoder1"):
    print(n, "gradient:", json.dumps(smoke["gradient_diagnostics"][n]))

G forwards: 1 autograd calls: 1
activation shapes: {'bridge': [1, 256, 32, 32], 'decoder1': [1, 24, 512, 512]}
enable_grad: True autocast_cuda_fp16: True no_grad_used: False inference_mode_used: False

bridge gradient: {"shape": [1, 256, 32, 32], "dtype": "torch.float32", "finite": true, "min": -0.01265224814414978, "max": 0.016281485557556152, "abs_max": 0.016281485557556152}
decoder1 gradient: {"shape": [1, 24, 512, 512], "dtype": "torch.float32", "finite": true, "min": -7.033348083496094e-06, "max": 5.066394805908203e-06, "abs_max": 7.033348083496094e-06}


## 8. Eight-stage numerical guard chain

In [9]:
print("guard stages:", gf["finite_guards_applied"])
for n in ("bridge","decoder1"):
    print(f"--- {n} ---")
    print(json.dumps(smoke["numerical_guard_chain"][n], indent=1))

guard stages:

 ['target_scalar_finite', 'gradient_finite', 'alpha_finite', 'pre_relu_weighted_map_finite', 'post_relu_raw_cam_finite', 'upsampled_prenorm_cam_finite', 'prenorm_cam_range_finite', 'normalized_cam_finite']
--- bridge ---
{
 "target_scalar_finite": true,
 "gradient_finite": true,
 "alpha_finite": true,
 "pre_relu_weighted_map_finite": true,
 "post_relu_raw_cam_finite": true,
 "upsampled_prenorm_cam_finite": true,
 "prenorm_cam_range_finite": true,
 "normalized_cam_finite": true,
 "alpha_abs_max": null,
 "pre_relu_weighted_map_abs_max": 0.013898851349949837,
 "post_relu_raw_cam_abs_max": 0.0,
 "upsampled_prenorm_cam": {
  "shape": [
   1,
   1,
   512,
   512
  ],
  "dtype": "torch.float32",
  "finite": true,
  "min": 0.0,
  "max": 0.0,
  "abs_max": 0.0
 },
 "upsampled_prenorm_cam_range": 0.0,
 "prenorm_cam_all_zero": true,
 "normalized_cam_min": 0.0,
 "normalized_cam_max": 0.0,
 "nonfinite_at_any_stage": false,
 "nan_to_num_applied": false,
 "clamp_to_hide_nonfinite": false
}
--- decode

## 9. CAM finiteness and zero-range treatment

In [10]:
cr = smoke["cam_results"]
print("bridge all-zero:", cr["bridge_cam_all_zero"], "| decoder1 all-zero:", cr["decoder1_cam_all_zero"])
print("finite zero-range:", cr["cams_finite_zero_range"], "| nonzero:", cr["cams_nonzero"])
print("finite_zero_range_cam_treatment:", cr["finite_zero_range_cam_treatment"])
print("cams:", json.dumps(cr["cams"], indent=1))

bridge all-zero: True | decoder1 all-zero: True
finite zero-range: True | nonzero: False
finite_zero_range_cam_treatment: PASS
cams: {
 "bridge": {
  "shape": [
   512,
   512
  ],
  "dtype": "torch.float32",
  "finite": true,
  "min": 0.0,
  "max": 0.0,
  "nonzero": false
 },
 "decoder1": {
  "shape": [
   512,
   512
  ],
  "dtype": "torch.float32",
  "finite": true,
  "min": 0.0,
  "max": 0.0,
  "nonzero": false
 }
}


## 10. Actual-same-forward empty-mask fallback check (load-bearing)

In [11]:
emc = smoke["empty_mask_actual_logits_check"]
for k in ("same_actual_G_logits_used","surrogate_zero_logits_used","empty_mask_rank","empty_mask_shape",
          "empty_mask_true_pixels","fallback_selected","helper_return","expected_full_plane_mean",
          "difference","atol","rtol","comparison_pass","finite","additional_G_forwards",
          "additional_autograd_calls"):
    print(f"{k}: {emc[k]}")

same_actual_G_logits_used: True
surrogate_zero_logits_used: False
empty_mask_rank: 2
empty_mask_shape: [512, 512]
empty_mask_true_pixels: 0
fallback_selected: True
helper_return: -13.863729476928711
expected_full_plane_mean: -13.863729476928711
difference: 0.0
atol: 1e-07
rtol: 0.0
comparison_pass: True
finite: True
additional_G_forwards: 0
additional_autograd_calls: 0


## 11. Direct parameter .grad audit and state immutability

In [12]:
pg = smoke["parameter_grad_state_audit"]
print("G parameter tensors:", pg["G"]["named_parameter_tensors"], "| non-null grads:", pg["G"]["non_null_grad_tensors"])
print("B_phi parameter tensors:", pg["B_phi"]["named_parameter_tensors"], "| non-null grads:", pg["B_phi"]["non_null_grad_tensors"])
print("NO_PARAMETER_GRAD_POPULATED:", pg["NO_PARAMETER_GRAD_POPULATED"])
print("mechanism:", pg["mechanism"])
print()
pi = smoke["parameter_immutability"]
print("G unchanged:", pi["G_state_tensors_unchanged"], "| B_phi unchanged:", pi["B_phi_state_tensors_unchanged"])
print("parameter_mutations:", pi["parameter_mutations"])

G parameter tensors: 86 | non-null grads: 0
B_phi parameter tensors: 11 | non-null grads: 0
NO_PARAMETER_GRAD_POPULATED: True
mechanism: torch.autograd.grad does not populate .grad; attribution targets are activations / numeric inputs, never parameters

G unchanged: True | B_phi unchanged: True
parameter_mutations: 0


## 12. Router synthetic smoke and zero-denominator policy

In [13]:
r = smoke["router_smoke"]
print("raw input:", r["raw_router_input_shape"], r["raw_router_input_dtype"], "finite:", r["raw_router_input_finite"])
print("normalized input:", r["normalized_router_input_shape"], "finite:", r["normalized_router_input_finite"])
print("delta_hat:", r["raw_delta_hat_shape"], "finite:", r["raw_delta_hat_finite"], "sigmoid_count:", r["sigmoid_count"])
print("gradient:", r["router_gradient_shape"], "finite:", r["router_gradient_finite"])
print("gradient x input:", r["gradient_x_input_shape"], "finite:", r["gradient_x_input_finite"])
print("task_id numeric attribution:", r["task_id_numeric_attribution"])
print()
z = smoke["zero_denominator_policy"]
print("denominator:", z["denominator"], "| shares:", z["normalized_group_shares"])
print("ZERO_ROUTER_GRAD_ATTRIBUTION:", z["ZERO_ROUTER_GRAD_ATTRIBUTION"],
      "| epsilon:", z["epsilon_added"], "| renormalization:", z["renormalization"],
      "| b_phi forwards used:", z["b_phi_forwards_used"])

raw input: [4, 202] float32 finite: True
normalized input: [4, 202] finite: True
delta_hat: [4] finite: True sigmoid_count: 0
gradient: [4, 202] finite: True
gradient x input: [4, 202] finite: True
task_id numeric attribution: False

denominator: 0.0 | shares: [None, None, None, None, None, None]
ZERO_ROUTER_GRAD_ATTRIBUTION: True | epsilon: False | renormalization: False | b_phi forwards used: 0


## 13. Execution counters, protected-data counters, scientific-output counters

In [14]:
ct = smoke["execution_counters"]
for k in ("G_model_constructions","B_phi_model_constructions","checkpoint_deserializations","strict_loads",
          "G_synthetic_forwards","B_phi_synthetic_forwards","total_model_forwards",
          "G_gradcam_autograd_calls","router_numeric_input_autograd_calls","total_autograd_calls",
          "synthetic_CAM_maps","synthetic_router_attribution_rows","synthetic_smoke_target_masks",
          "R_psi_model_constructions","S_omega_model_constructions","optimizer_constructions",
          "optimizer_step_calls","training_backward_calls","parameter_mutations"):
    print(f"{k}: {ct[k]}")
print()
print("protected data:", json.dumps(smoke["protected_data_counters"]))
print("scientific outputs:", json.dumps(smoke["scientific_output_counters"]))
print("counters_all_protected_zero:", smoke["counters_all_protected_zero"])

G_model_constructions: 1
B_phi_model_constructions: 1
checkpoint_deserializations: 2
strict_loads: 2
G_synthetic_forwards: 1
B_phi_synthetic_forwards: 1
total_model_forwards: 2
G_gradcam_autograd_calls: 1
router_numeric_input_autograd_calls: 1
total_autograd_calls: 2
synthetic_CAM_maps: 2
synthetic_router_attribution_rows: 4
synthetic_smoke_target_masks: 2
R_psi_model_constructions: 0
S_omega_model_constructions: 0
optimizer_constructions: 0
optimizer_step_calls: 0
training_backward_calls: 0
parameter_mutations: 0

protected data: {"raw_VAL_image_reads": 0, "real_persisted_VAL_probability_reads": 0, "VAL_GT_reads": 0, "router_feature_cache_rows_read": 0, "ROUTER_HOLDOUT_reads": 0, "TEST_reads": 0, "TRAIN_raw_reads": 0}
scientific outputs: {"scientific_outputs": 0, "scientific_figures": 0, "scientific_tables": 0, "scientific_gradcam_outputs": 0, "scientific_router_attribution_outputs": 0, "scientific_film_outputs": 0}
counters_all_protected_zero: True


## 14. Governance state

In [15]:
print("classification:", smoke["classification"])
print("not:", smoke["not"])
print("gates:", json.dumps(smoke["gates"]))
print("phase35_scientific_execution_started:", smoke["phase35_scientific_execution_started"])
print("phase36_started:", smoke["phase36_started"])
print("TEST:", smoke["TEST"])
print("ROUTER_HOLDOUT:", smoke["ROUTER_HOLDOUT"])
print()
print("terminal:", term["terminal"])

classification:

 PRE_SCIENTIFIC_SYNTHETIC_NUMERICAL_SAFETY_AND_INTERFACE_VALIDATION
not: NOT_PHASE35_SCIENTIFIC_INTERPRETABILITY_RESULT
gates: {"PHASE35_SMOKE_EXECUTE_set": true, "PHASE35_EXECUTE_set": false, "smoke_gate_env": "PHASE35_SMOKE_EXECUTE", "science_gate_env": "PHASE35_EXECUTE", "mode": "smoke"}
phase35_scientific_execution_started: False
phase36_started: False
TEST: LOCKED_UNTIL_PHASE39
ROUTER_HOLDOUT: OPENED_AND_FROZEN

terminal: PHASE35_SYNTHETIC_SMOKE_V1_0F_BLOCKED__CANONICAL_MODEL_OR_NUMERICAL_INTERPRETABILITY_INTERFACE_CHECK_FAILED__ZERO_PROTECTED_DATA_OR_SCIENTIFIC_OUTPUT_ACCESS


## 15. Reviewer summary

- The frozen v1.0f source/hash gate passed for all 18 frozen sources BEFORE model construction.
- Canonical `G` and `B_phi` constructed once each, strict-loaded with 0 missing / 0 unexpected keys,
  at 3,523,001 and 36,801 parameters respectively; `R_psi`/`S_omega` untouched.
- Exactly one synthetic `G` forward and one router `B_phi` forward were run; two autograd calls total.
- All eight numerical guard stages are finite for both `bridge` and `decoder1`; no `nan_to_num` or
  hidden clamping was used. Both CAMs are finite zero-range (allowed PASS).
- The load-bearing actual-same-forward empty-mask fallback check passed with difference `0.0` at
  `atol=1e-7, rtol=0`, using the ACTUAL logits; no surrogate zero logits were used.
- Direct parameter `.grad` audit: 0 non-null gradients for both models; state identity unchanged.
- Protected-data and scientific-output counters are all zero.

**The smoke correctly FAILED CLOSED** on a single internal execution-count integrity mismatch
(`synthetic_smoke_target_masks` = 2 vs expected 1). This is a bookkeeping defect in the v1.0f smoke
harness's expected-counter declaration, not a model, numerical, interface, protected-data, or
scientific defect. Phase35 scientific execution remains NOT STARTED and is NOT eligible until this
counter contract is reconciled. TEST remains `LOCKED_UNTIL_PHASE39`.